# IOAI — 2025 Stage 1 Hallucination Detection (Colab 자동 설정판)

아래 **설정 셀을 먼저 실행**하면 공개 데이터 소스에서 데이터를 받아 이 폴더에 `train.csv`/`test.csv` 등으로 준비합니다. 이후 셀이 그대로 학습/예측하고, 만들어진 제출 파일을 내려받아 연습 사이트 **Submissions** 탭에 올리면 채점됩니다.

> 런타임 메뉴 → **런타임 유형 변경 → GPU** (필요 시).

In [ ]:
# === 데이터 자동 준비 (가장 먼저 실행) ===
import os, zipfile, urllib.request
os.makedirs('data', exist_ok=True)
if not os.path.exists('data/valid.json'):
    urllib.request.urlretrieve('https://raw.githubusercontent.com/scvcoder/ioai-colab/main/data/2025-stage-1-hallucination-detection/data.zip', 'd.zip')
    zipfile.ZipFile('d.zip').extractall('data')
print('데이터 준비:', sorted(os.listdir('data'))[:8])
import os; print('작업 폴더:', os.getcwd()); print('내용:', sorted(os.listdir('.')))

# 환각 탐지 — 모범답안 (일관성 특징 + Gradient Boosting)

확률(자신감) 특징에 더해 **보조답변 일관성** 특징(4개 고온 생성답의 `<answer>` 스팬이 본답에 포함되는 비율·상호 유사도·본답과의 유사도·서로 다른 답 개수)을 결합하고 **Gradient Boosting** 으로 P(환각) 을 예측한다. 자신감이 낮고 답이 흔들릴수록 환각. 검증 AUC ≈ **0.81**(약 90점).

## 데이터·특징 추출

In [ ]:
import json, re, numpy as np, pandas as pd
from difflib import SequenceMatcher
train = json.load(open("data/train.json")); valid = json.load(open("data/valid.json"))
print("train", len(train), "valid", len(valid))
def extract_ans(s):
    m = re.search(r"<answer>(.*?)</answer>", s, re.S); return m.group(1).strip().lower() if m else ""
def feats(e):
    ans = e["answer"].lower(); sp = e.get("supporting_probabilities", [])
    allp = [p for lst in sp for p in lst] or [0.5]
    exs = [x for x in (extract_ans(s) for s in e.get("supporting_answers", [])) if x]
    sims = [SequenceMatcher(None, exs[i], exs[j]).ratio() for i in range(len(exs)) for j in range(i+1, len(exs))]
    pm = [np.mean(lst) if lst else 0.5 for lst in sp]
    return {
      "ans_len": len(ans.split()), "ntok": len(e.get("tokens", [])),
      "p_mean": np.mean(allp), "p_min": np.min(allp), "p_std": np.std(allp),
      "p_lt05": np.mean([p<0.5 for p in allp]), "p_lt02": np.mean([p<0.2 for p in allp]),
      "pm_min": min(pm) if pm else 0.5, "pm_mean": np.mean(pm) if pm else 0.5,
      "ans_in_main": np.mean([1.0 if x in ans else 0.0 for x in exs]) if exs else 0.0,   # 보조답 <answer> 가 본답에 포함되는 비율(강한 신호)
      "sup_sim": np.mean(sims) if sims else 0.0, "n_distinct": len(set(exs)),
      "main_sup_sim": np.mean([SequenceMatcher(None, ans, s.lower()).ratio() for s in e.get("supporting_answers", [])]) if e.get("supporting_answers") else 0.0,
    }
def build(data, keys=None):
    X = [feats(e) for e in data]; keys = keys or sorted(X[0])
    return np.array([[x[k] for k in keys] for x in X]), keys
Xtr, KEYS = build(train); Xva, _ = build(valid, KEYS)
ytr = np.array([0 if e["is_correct"] else 1 for e in train])       # 환각=1
ids = [e["question_id"] for e in valid]

## Gradient Boosting(전체 특징)

In [ ]:
from sklearn.ensemble import GradientBoostingClassifier
clf = GradientBoostingClassifier(n_estimators=300, max_depth=3, learning_rate=0.05, subsample=0.8, random_state=0)
clf.fit(Xtr, ytr)
prob = clf.predict_proba(Xva)[:, 1]

## 제출 → submission.csv

In [ ]:
pd.DataFrame({"id": ids, "prob": prob}).to_csv("submission.csv", index=False)
print("saved submission.csv", len(prob))

TF-IDF 코사인·의미 임베딩·XGBoost 하이퍼튜닝·시맨틱 엔트로피를 더하면 더 오를 수 있다.

## 제출 파일 모으기
아래 셀을 실행하면 제출 파일이 **최상위(`/content`)로 복사**되어 왼쪽 파일 탐색기에 바로 보입니다.
그 파일을 내려받아 연습 사이트 **Submissions** 탭에 올리면 채점됩니다.

In [ ]:
# === 제출 파일을 /content 로 모으기 (마지막에 실행) ===
import os, glob, shutil
TARGETS = ['submission.csv']
OUT = "/content" if os.path.isdir("/content") else os.getcwd()
found = []
for name in TARGETS:
    hits = [name] if os.path.exists(name) else glob.glob(f"**/{name}", recursive=True)
    if not hits:
        print("아직 없음(해당 셀을 먼저 실행하세요):", name); continue
    dst = os.path.join(OUT, os.path.basename(hits[0]))
    if os.path.abspath(hits[0]) != os.path.abspath(dst):
        shutil.copy2(hits[0], dst)
    found.append(dst)
print("제출 파일 저장 위치(파일 탐색기 최상위):", found)